In [ ]:
# !pip install apify_client deep_translator

In [ ]:
import math
import re
from urllib.parse import urlparse
from datetime import datetime
import os
import traceback
import numpy as np
import pandas as pd
from apify_client import ApifyClient
from deep_translator import GoogleTranslator


# =============================================
# CONFIGURATION — edit these before running
# =============================================
APIFY_TOKEN = "apify_api_OfHNt7JKFsBMjMWywHCwizhXWztDxU4C2oqs"

# Actor: axesso_data/amazon-reviews-scraper
# https://apify.com/axesso_data/amazon-reviews-scraper
# Same actor you were using — ID is ZebkvH3nVOrafqr5T.
# This actor supports: domainCode, sortBy, filterByStar, formatType, maxPages, variationId output.
ACTOR_ID = "ZebkvH3nVOrafqr5T"

INPUT_CSV = r"Amazon group 3 input.csv"
OUTPUT_FOLDER = r"Amazon CA"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

INPUT_URL_COLUMN = "url"
PCODE_COLUMN     = "p_code"
NAME_COLUMN      = "name"
REGION_COLUMN    = "region"

# Pages per star bucket (1 page = ~10 reviews on Amazon)
MAX_PAGES_PER_BUCKET = 1
STAR_BUCKETS = ["one_star", "two_star", "three_star", "four_star", "five_star"]
MAX_REVIEWS_PER_PRODUCT = 500

# KEY FIX: pass "current_format" so the actor returns variationId reliably.
# Set to None to revert to old behaviour.
FORMAT_TYPE = "current_format"
SORT_BY     = "recent"

gt = GoogleTranslator(source="auto", target="en")
date_of_scraping = datetime.today().strftime("%Y-%m-%d")

# =============================================
# DOMAIN → ACCEPTED DATE-STRING PREFIXES
#
# Amazon writes the reviewer's country into the
# 'date' field.  The exact format depends on the
# marketplace language.  Known formats:
#
#  EN  Reviewed in Canada on February 3, 2025
#  EN  Reviewed in the United States on March 15, 2026
#  EN  Reviewed in the United Kingdom on 28 November 2024
#  EN  Reviewed in Belgium on 9 July 2025
#  FR  Avis laissé en France le 4 mars 2026
#  FR  Commenté en Belgique le 8 février 2025
#  DE  Bewertet in Deutschland am 28. September 2025
#  IT  Recensito in Italia il 8 maggio 2024
#  ES  Reseñado en España el 31 de julio de 2025
#  NL  Beoordeeld in Nederland op 8 mei 2025
#  PL  Zrecenzowano w Polsce w dniu 29 kwietnia 2026
#
# Each entry is a list of re.match() patterns
# (anchored at the start, case-insensitive).
# A review passes ONLY if its date string matches
# one of the patterns exactly — no fail-open.
# Unknown / empty / unparseable dates are REJECTED.
# =============================================
DOMAIN_ACCEPTED_PREFIXES = {
    "com":    [r"reviewed in the united states on"],
    "ca":     [r"reviewed in canada on"],
    "co.uk":  [r"reviewed in the united kingdom on"],
    "de":     [r"bewertet in deutschland am",
               r"reviewed in germany on"],
    "fr":     [r"avis laiss[eé] en france le",
               r"reviewed in france on"],
    "it":     [r"recensito in italia il",
               r"reviewed in italy on"],
    "es":     [r"rese[nñ]ado en espa[nñ]a el",
               r"reviewed in spain on"],
    "nl":     [r"beoordeeld in nederland op",
               r"reviewed in (?:the )?netherlands on"],
    "pl":     [r"zrecenzowano w polsce w dniu",
               r"reviewed in poland on"],
    "com.be": [r"comment[eé] en belgique le",
               r"reviewed in belgium on"],
    "co.jp":  [r"reviewed in japan on"],
    "com.au": [r"reviewed in australia on"],
    "com.br": [r"reviewed in brazil on"],
    "se":     [r"reviewed in sweden on"],
    "ae":     [r"reviewed in (?:the )?united arab emirates on"],
    "com.mx": [r"reviewed in mexico on"],
}

# Pre-compile all patterns once at import time
_COMPILED_PREFIXES = {
    domain: [re.compile(p, re.IGNORECASE) for p in patterns]
    for domain, patterns in DOMAIN_ACCEPTED_PREFIXES.items()
}

# Normalises variation attribute keys to canonical column names
MAP = {
    "pattern name": "Style",
    "pattern":      "Style",
    "style name":   "Style",
    "style":        "Style",
    "color name":   "Color",
    "colour":       "Color",
    "colour name":  "Color",
    "color":        "Color",
    "size name":    "Size",
    "size":         "Size",
}

# =============================================
# REGION → DOMAIN CODE LOOKUP
# Maps the value in the CSV's 'region' column
# to the domainCode the actor expects.
# The CSV can contain any subset of these regions;
# only the ones present in the file will run.
# =============================================
REGION_DOMAIN_MAP = {
    "US": "com",
    "UK": "co.uk",
    "DE": "de",
    "FR": "fr",
    "IT": "it",
    "ES": "es",
    "BE": "com.be",
    "NL": "nl",
    "PL": "pl",
    "CA": "ca",
    "JP": "co.jp",
    "AU": "com.au",
    "BR": "com.br",
    "SE": "se",
    "AE": "ae",
    "MX": "com.mx",
}


# =============================================
# HELPERS
# =============================================
def translate_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return x
    if isinstance(x, (list, tuple)):
        return [translate_cell(i) for i in x]
    text = str(x).strip()
    if not text:
        return x
    try:
        return gt.translate(text)
    except Exception:
        return x


def translate_dataframe(df):
    try:
        return df.map(translate_cell)
    except AttributeError:
        return df.applymap(translate_cell)


def extract_asin_from_url(url: str):
    """Pull the 10-char ASIN out of any Amazon product URL."""
    m = re.search(r"/(?:dp|gp/product|gp/aw/d|product-reviews)/([A-Z0-9]{10})", url, re.I)
    return m.group(1) if m else None


def _is_valid(val):
    if val is None:
        return False
    try:
        if pd.isna(val):
            return False
    except Exception:
        pass
    return not (isinstance(val, str) and not val.strip())


def slugify(value: str, max_len: int = 80) -> str:
    if not value:
        return ""
    value = re.sub(r"[^A-Za-z0-9\s]+", "_", value).strip("_")
    return (value[:max_len].rstrip("_") if len(value) > max_len else value) or "item"


# =============================================
# APIFY FETCH
# =============================================
def fetch_amazon_reviews(
    asin,
    domain_code,
    format_type=None,
    sort_by="recent",
    max_pages=1,
    stars=None,
    actor_id=ACTOR_ID,
    apify_token=APIFY_TOKEN,
):
    """
    Call axesso_data/amazon-reviews-scraper.

    The actor input schema wraps everything inside an 'input' array.
    Setting formatType='current_format' ensures the variationId field
    is populated in the output — critical for variant-specific filtering.
    """
    if not apify_token:
        raise RuntimeError("APIFY_TOKEN is empty.")

    client = ApifyClient(apify_token)

    review_input = {
        "asin":       asin,
        "domainCode": domain_code,
        "sortBy":     sort_by,
        "maxPages":   int(max_pages),
    }

    if _is_valid(format_type):
        review_input["formatType"] = str(format_type).strip()
    if _is_valid(stars):
        review_input["filterByStar"] = str(stars).strip()

    run = client.actor(actor_id).call(run_input={"input": [review_input]})

    dataset_id = run.get("defaultDatasetId")
    if not dataset_id:
        raise RuntimeError("No dataset returned — check Apify Console actor logs.")

    items = list(client.dataset(dataset_id).iterate_items())
    for it in items:
        it.setdefault("domainCode", domain_code)
    return items


# =============================================
# VARIATION PARSER
# =============================================
def parse_variations(x):
    out = {}
    if isinstance(x, (list, tuple)):
        for item in x:
            if isinstance(item, str) and ":" in item:
                k, v = (s.strip() for s in item.split(":", 1))
                k = MAP.get(k.lower(), k)
                if v:
                    out.setdefault(k, v)
    return out


# =============================================
# ENRICH REVIEWS
# =============================================
def enrich_reviews(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Expand variation attributes into columns
    try:
        df = df.join(
            df.get("variationList", pd.Series([[]] * len(df), index=df.index))
              .apply(parse_variations)
              .apply(pd.Series)
        )
    except Exception:
        pass

    df["P_code"]         = df.get("asin")
    df["Review"]         = df.get("text")
    df["Title_clean"]    = df.get("title")
    df["Username_clean"] = df.get("userName")

    count_reviews = df.get("countReviews", pd.Series([pd.NA] * len(df), index=df.index))
    df["Total_Reviews"] = pd.to_numeric(count_reviews, errors="coerce").astype("Int64")

    product_rating = df.get("productRating", pd.Series([pd.NA] * len(df), index=df.index))
    df["Average_Rating"] = pd.to_numeric(
        product_rating.astype(str).str.extract(r"(\d+\.?\d*)")[0], errors="coerce"
    )

    rating_col = df.get("rating", pd.Series([pd.NA] * len(df), index=df.index))
    df["Ratings_Num"] = pd.to_numeric(
        rating_col.astype(str).str.extract(r"(\d+)")[0], errors="coerce"
    ).astype("Int64")

    df["RatingLabel"] = np.where(
        df["Ratings_Num"].notna(),
        df["Ratings_Num"].astype(str) + " Star",
        pd.NA,
    )

    date_col = df.get("date", pd.Series([""] * len(df), index=df.index))
    df["Country_Name"] = date_col.astype(str).str.extract(r"Reviewed in (?:the )?(.+?) on")[0]

    verified_raw = df.get("verified", pd.Series([False] * len(df), index=df.index))
    vine_raw     = df.get("vine",     pd.Series([False] * len(df), index=df.index))
    verified_bool = verified_raw.replace({"True": True, "False": False}).fillna(False).astype(bool)
    vine_bool     = vine_raw.replace({"True": True, "False": False}).fillna(False).astype(bool)

    df["Badge"] = np.where(verified_bool, "Verified Purchase", "")
    df["Reviewer_Badges"] = [
        " | ".join(filter(None, [
            "Verified Purchase" if v else "",
            "Vine" if vn else ""
        ]))
        for v, vn in zip(verified_bool, vine_bool)
    ]

    num_help = pd.to_numeric(
        df.get("numberOfHelpful", pd.Series([0] * len(df), index=df.index)),
        errors="coerce"
    ).fillna(0).astype(int)

    df["HelpfulText"] = np.where(
        num_help == 0, "",
        np.where(num_help == 1,
                 "1 person found this helpful",
                 num_help.astype(str) + " people found this helpful")
    )

    df["WithReview"]    = np.where(df["Review"].astype(str).str.strip().ne(""), "Yes", "No")
    df["DateOfScraping"] = date_of_scraping

    if "reviewId" in df.columns:
        domain_code = df.get("domainCode", pd.Series(["com"] * len(df), index=df.index)).fillna("com").astype(str)
        df["Post_Link"] = "https://www.amazon." + domain_code + "/gp/customer-reviews/" + df["reviewId"].astype(str)
    else:
        df["Post_Link"] = ""

    if "profilePath" in df.columns:
        domain_code  = df.get("domainCode", pd.Series(["com"] * len(df), index=df.index)).fillna("com").astype(str)
        profile_path = df.get("profilePath", pd.Series([""] * len(df), index=df.index)).fillna("").astype(str)
        df["Profile_URL"] = np.where(
            profile_path.str.strip().ne(""),
            "https://www.amazon." + domain_code + profile_path,
            "",
        )
    else:
        df["Profile_URL"] = ""

    return df


# =============================================
# VARIANT FILTER
# =============================================
def apply_variant_filter(df_raw: pd.DataFrame, asin: str) -> pd.DataFrame:
    """
    Keep only rows where variationId matches the target ASIN.

    Strategy:
    1. If the 'variationId' column exists AND is non-null for at least half the rows,
       filter strictly by ASIN match.
    2. If variationId is mostly null/missing (actor didn't populate it), warn and keep all.
    3. If the column is absent entirely, warn and keep all.
    """
    if "variationId" not in df_raw.columns:
        print("  ⚠  'variationId' column absent — cannot filter by variant. "
              "Try setting FORMAT_TYPE='current_format'.")
        return df_raw

    non_null_ratio = df_raw["variationId"].notna().mean()
    if non_null_ratio < 0.5:
        print(f"  ⚠  variationId mostly null ({non_null_ratio:.0%} populated) — "
              "skipping variant filter. Check FORMAT_TYPE setting.")
        return df_raw

    before = len(df_raw)
    filtered = df_raw[df_raw["variationId"].astype(str).str.upper() == asin.upper()]
    after = len(filtered)
    print(f"  Variant filter (variationId == {asin}): {before} → {after} reviews")

    if after == 0:
        print("  ⚠  No reviews matched this variant ASIN. "
              "The product may not have variant-level review pages, "
              "or the ASIN is a parent ASIN. Returning all reviews.")
        return df_raw

    return filtered


# =============================================
# LOCATION FILTER
# =============================================
def apply_location_filter(df_raw: pd.DataFrame, domain_code: str):
    """
    Keep only reviews whose 'date' field starts with one of the
    known accepted prefixes for this marketplace (see DOMAIN_ACCEPTED_PREFIXES).

    Strict mode — no fail-open:
      - Foreign country  → rejected
      - Empty / NaN      → rejected
      - Unknown format   → rejected

    Returns (filtered_df, n_kept, n_dropped).
    """
    compiled = _COMPILED_PREFIXES.get(domain_code)
    if not compiled:
        print(f"  ⚠  No prefix patterns for domainCode '{domain_code}' — "
              "location filter skipped. Add it to DOMAIN_ACCEPTED_PREFIXES.")
        return df_raw, len(df_raw), 0

    if "date" not in df_raw.columns:
        print("  ⚠  'date' column absent — location filter skipped.")
        return df_raw, len(df_raw), 0

    def _matches(date_val) -> bool:
        s = str(date_val).strip()
        if not s or s.lower() == "nan":
            return False                       # empty/null → reject
        return any(pat.match(s) for pat in compiled)

    mask_keep = df_raw["date"].apply(_matches)

    before   = len(df_raw)
    filtered = df_raw[mask_keep]
    dropped  = before - len(filtered)

    accepted_display = DOMAIN_ACCEPTED_PREFIXES.get(domain_code, [])
    print(f"  Location filter (domainCode='{domain_code}'): "
          f"{before} → {len(filtered)} ({dropped} rejected)")
    if dropped:
        # Show a sample of what was rejected so it's easy to diagnose
        rejected_samples = (
            df_raw.loc[~mask_keep, "date"]
            .dropna().astype(str).unique()[:3]
        )
        for s in rejected_samples:
            print(f"    rejected sample: {s!r}")

    return filtered, len(filtered), dropped


# =============================================
# PROCESS ONE REGION'S ROWS
# =============================================
def process_region_rows(
    links_df: pd.DataFrame,
    region_cfg: dict,
    failures: list,
    successes: list,
    skipped: list,
):
    region_code = region_cfg["code"]
    domain_code = region_cfg["domainCode"]   # e.g. "de", "co.uk", "com"

    for idx, row in links_df.iterrows():
        try:
            url_val = row.get(INPUT_URL_COLUMN)
            if pd.isna(url_val):
                reason = "URL is NaN"
                print(f"[{region_code} Row {idx}] {reason} — skipping.")
                skipped.append({
                    "region_config": region_code, "row_index": idx,
                    "input_url": url_val, "asin": None,
                    "name": row.get(NAME_COLUMN), "reason": reason,
                })
                continue

            url = str(url_val).strip()
            if not url.startswith("http"):
                reason = f"Invalid URL: '{url}'"
                print(f"[{region_code} Row {idx}] {reason} — skipping.")
                skipped.append({
                    "region_config": region_code, "row_index": idx,
                    "input_url": url, "asin": None,
                    "name": row.get(NAME_COLUMN), "reason": reason,
                })
                continue

            product_name   = str(row.get(NAME_COLUMN,   "") or "").strip()
            row_region_val = str(row.get(REGION_COLUMN, "") or "").strip()

            print(f"\n--- {region_code} row {idx}: {url} ---")
            if product_name:   print(f"  Product name : {product_name}")
            if row_region_val: print(f"  Region (row) : {row_region_val}")

            # ASIN: prefer sheet column, fall back to URL
            asin_from_sheet = row.get(PCODE_COLUMN) if PCODE_COLUMN in links_df.columns else None
            asin = str(asin_from_sheet).strip() if _is_valid(asin_from_sheet) else extract_asin_from_url(url)

            if not asin:
                reason = "Could not determine ASIN from URL or sheet column"
                print(f"  ✗ {reason} — skipping.")
                skipped.append({
                    "region_config": region_code, "row_index": idx,
                    "input_url": url, "asin": None,
                    "name": product_name, "reason": reason,
                })
                continue

            print(f"  Region: {region_code} | DomainCode: {domain_code} | ASIN: {asin}")

            # Fetch star buckets
            product_items = []
            for star in STAR_BUCKETS:
                print(f"  → Fetching {star}…", end=" ", flush=True)
                try:
                    batch = fetch_amazon_reviews(
                        asin=asin,
                        domain_code=domain_code,
                        format_type=FORMAT_TYPE,
                        sort_by=SORT_BY,
                        max_pages=MAX_PAGES_PER_BUCKET,
                        stars=star,
                    )
                    print(f"{len(batch)} reviews")
                    product_items.extend(batch)
                except Exception as e:
                    print(f"ERROR: {e}")
                    continue

            if not product_items:
                reason = "Actor returned zero reviews across all star buckets"
                print(f"  ✗ {reason} — skipping.")
                skipped.append({
                    "region_config": region_code, "row_index": idx,
                    "input_url": url, "asin": asin,
                    "name": product_name, "reason": reason,
                })
                continue

            df_raw = pd.DataFrame(product_items)

            # ---- Variant filter ----
            df_raw = apply_variant_filter(df_raw, asin)

            # ---- Location filter: drop reviews from other countries ----
            df_raw, n_kept, n_dropped = apply_location_filter(df_raw, domain_code)

            if df_raw.empty:
                reason = (
                    f"All {n_dropped + n_kept} reviews were from foreign marketplaces "
                    f"(domainCode='{domain_code}') — no reviews from the correct country"
                )
                print(f"  ✗ {reason} — discarding product.")
                skipped.append({
                    "region_config": region_code, "row_index": idx,
                    "input_url": url, "asin": asin,
                    "name": product_name, "reason": reason,
                })
                continue

            # ---- Deduplication ----
            if "reviewId" in df_raw.columns:
                before = len(df_raw)
                df_raw = df_raw.drop_duplicates(subset=["reviewId"])
                print(f"  Dedup by reviewId: {before} → {len(df_raw)}")

            df_raw = df_raw.head(MAX_REVIEWS_PER_PRODUCT)
            print(f"  Keeping {len(df_raw)} reviews (cap: {MAX_REVIEWS_PER_PRODUCT})")

            final_df = enrich_reviews(df_raw)
            final_df["Input_URL"]    = url
            final_df["Region"]       = row_region_val if row_region_val else region_code
            final_df["Product_Name"] = product_name
            final_df["Input_P_code"] = asin_from_sheet if _is_valid(asin_from_sheet) else asin

            for col in links_df.columns:
                if col != INPUT_URL_COLUMN:
                    final_df[col] = row.get(col)

            rename_map = {
                "productRating":   "Product Rating",
                "countReviews":    "Review_Count",
                "vine":            "Is_Vine",
                "date":            "Review_Date",
                "Title_clean":     "Title",
                "Username_clean":  "Username",
                "Ratings_Num":     "Rating (Num)",
                "RatingLabel":     "Rating",
                "Country_Name":    "Location",
                "Reviewer_Badges": "Badges",
                "Post_Link":       "Post Link",
            }
            final_df = final_df.rename(columns=rename_map)

            desired_cols = [
                "Product Rating", "Review_Count", "Is_Vine", "Review_Date",
                "P_code", "Review", "Title", "Username",
                "Rating (Num)", "Rating", "Location", "Badges",
                "Post Link", "variationId", "variationList",
            ]
            final_df = final_df[[c for c in desired_cols if c in final_df.columns]]

            filename = f"{product_name}.csv" if product_name else f"{asin}_row{idx}.csv"
            out_path = os.path.join(OUTPUT_FOLDER, filename)
            final_df.to_csv(out_path, index=False)
            print(f"  ✔ Saved {len(final_df)} reviews → '{out_path}'")

            successes.append({
                "region_config":  region_code,
                "row_index":      idx,
                "input_url":      url,
                "asin":           asin,
                "output_file":    out_path,
                "reviews_saved":  len(final_df),
            })

        except Exception as e:
            tb = traceback.format_exc()
            failures.append({
                "region_config":  region_cfg.get("code"),
                "row_index":      idx,
                "input_url":      row.get(INPUT_URL_COLUMN),
                "p_code":         row.get(PCODE_COLUMN),
                "name":           row.get(NAME_COLUMN),
                "region_in_row":  row.get(REGION_COLUMN),
                "error":          repr(e),
                "traceback":      tb,
            })
            print(f"[FAILED] {region_code} Row {idx} | URL={row.get(INPUT_URL_COLUMN)} | Error={e}")


# =============================================
# MAIN
# =============================================
def main():
    failures, successes, skipped = [], [], []

    all_df = pd.read_csv(INPUT_CSV)
    print(f"Loaded '{INPUT_CSV}' — {len(all_df)} rows")

    for col in (INPUT_URL_COLUMN, REGION_COLUMN):
        if col not in all_df.columns:
            raise ValueError(
                f"Required column '{col}' not found. "
                f"Available columns: {all_df.columns.tolist()}"
            )

    # Derive the unique regions present in the file, preserving order
    regions_in_file = all_df[REGION_COLUMN].dropna().astype(str).str.strip().str.upper().unique()
    print(f"Regions found in file: {list(regions_in_file)}")

    for region_code in regions_in_file:
        domain_code = REGION_DOMAIN_MAP.get(region_code)
        if not domain_code:
            print(f"\n[{region_code}] No domainCode mapping — skipping. "
                  f"Add it to REGION_DOMAIN_MAP.")
            continue

        links_df = all_df[
            all_df[REGION_COLUMN].astype(str).str.strip().str.upper() == region_code
        ].copy()

        print(f"\n{'='*55}")
        print(f" Region: {region_code} | DomainCode: {domain_code} | Rows: {len(links_df)}")
        print(f"{'='*55}")

        region_cfg = {"code": region_code, "domainCode": domain_code}
        process_region_rows(links_df, region_cfg, failures, successes, skipped)

    # ---- Reports ----
    # Named after all regions that ran, e.g. SUCCESS_ROWS_DE_FR.csv
    run_code = "_".join(regions_in_file) if len(regions_in_file) else "RUN"

    if successes:
        path = os.path.join(OUTPUT_FOLDER, f"SUCCESS_ROWS_{run_code}.csv")
        pd.DataFrame(successes).to_csv(path, index=False)
        print(f"\n✔ Success report : {path} ({len(successes)} rows)")

    if skipped:
        path = os.path.join(OUTPUT_FOLDER, f"SKIPPED_ROWS_{run_code}.csv")
        pd.DataFrame(skipped).to_csv(path, index=False)
        print(f"⏭  Skipped report : {path} ({len(skipped)} rows)")
        print("   Skipped reasons:")
        for r in pd.DataFrame(skipped).groupby("reason", sort=False).size().items():
            print(f"     {r[1]:>3}x  {r[0]}")
    else:
        print("\n✔ No intentionally skipped rows.")

    if failures:
        path = os.path.join(OUTPUT_FOLDER, f"FAILED_ROWS_{run_code}.csv")
        pd.DataFrame(failures).to_csv(path, index=False)
        print(f"✗ Failure report  : {path} ({len(failures)} rows)")
    else:
        print("✔ No unexpected failures.")

    print("\nAll done.")


if __name__ == "__main__":
    main()


## Output Summary
Run the cell below after scraping to see a per-region breakdown of how many reviews were saved.

In [ ]:
import os, re, pandas as pd
from collections import defaultdict

region_summary = defaultdict(lambda: {"files": [], "total_records": 0})

for filename in os.listdir(OUTPUT_FOLDER):
    if not filename.lower().endswith(".csv"):
        continue
    m = re.search(r'_([A-Za-z]+)\.csv$', filename)
    region = m.group(1) if m else "Unknown"
    try:
        df   = pd.read_csv(os.path.join(OUTPUT_FOLDER, filename))
        cnt  = len(df)
        region_summary[region]["files"].append((filename, cnt))
        region_summary[region]["total_records"] += cnt
    except Exception as e:
        print(f"Error reading {filename}: {e}")

for region, data in region_summary.items():
    print(f"\n🌍 {region}")
    for f, c in data["files"]:
        print(f"   {f}: {c} records")
    print(f"   ➤ Total: {data['total_records']}")
